In [ ]:
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
import numpy as np
from sklearn.decomposition import NMF
from scipy.ndimage import median_filter, gaussian_filter
from scipy.interpolate import RegularGridInterpolator
from pyFAI.integrator.azimuthal import AzimuthalIntegrator
from pyFAI.calibrant import CALIBRANT_FACTORY
import concurrent.futures
from tqdm.auto import tqdm

### 1. Load the experimental dataset

In [ ]:
# load the experimental dataset
DATASET = r""

with h5py.File(DATASET, "r") as f:
    patterns = np.array(f["train/images"])
    labels = np.array(f["train/labels"])
    print(f"Loaded {len(patterns)} patterns with shape {patterns[0].shape}")

In [ ]:
# plot a few example patterns
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for i, img in enumerate(patterns[:4]):
    axes[i].imshow(img, cmap="viridis", origin="lower", norm=SymLogNorm(linthresh=1))
    axes[i].set_title(f"Pattern {i+1}")
    axes[i].axis("off")

### 2. Background Extraction

To extract the background from the experimental dataset, we will perform azimuthal integration and median filtering on each pattern. This will help us isolate the background signal from the diffraction patterns. The extracted backgrounds are then learned using Non-negative Matrix Factorization (NMF) to identify the underlying components that contribute to the background, which can be randomly sampled to generate gradient backgrounds for training data synthesis.

In [ ]:
# perform azimuthal integration and median filtering to get the background for each pattern

WAVELENGTH = 0.5121197787410171e-10  # X-ray wavelength in meters
PIXEL_SIZE = 75e-6  # Pixel size in meters
DETECTOR = "Eiger2Cdte_1M"  # Detector type

backgrounds = np.zeros_like(patterns)

def process_single_frame(i):
    """Function to process one frame so we can run them in parallel."""
    img = patterns[i]
    
    ai = AzimuthalIntegrator(
        dist=labels[i][0], 
        poni1=labels[i][1], 
        poni2=labels[i][2],
        rot1=labels[i][3],
        rot2=labels[i][4],
        rot3=labels[i][5],
        pixel1=PIXEL_SIZE,
        pixel2=PIXEL_SIZE,
        wavelength=WAVELENGTH,
        detector=DETECTOR
    )

    polar, q_bins, chi_bins = ai.integrate2d(
        img,
        npt_rad=1500,      
        npt_azim=3600,     
        unit="q_A^-1",
        method="bbox", 
        correctSolidAngle=False,
        polarization_factor=None,
    )

    filtered_polar = median_filter(polar, size=(1, 251))
    smoothed_polar = gaussian_filter(filtered_polar, sigma=(20, 3))

    q_map = ai.array_from_unit(unit="q_A^-1")
    chi_map = ai.array_from_unit(unit="chi_rad") 

    interpolator = RegularGridInterpolator(
        (chi_bins, q_bins), 
        smoothed_polar, 
        bounds_error=False, 
        fill_value=0
    )

    chi_map_degrees = np.degrees(chi_map)
    coords = np.stack([chi_map_degrees.ravel(), q_map.ravel()], axis=-1)
    
    polar_reconstructed = interpolator(coords).reshape(img.shape)
    polar_smoothed = gaussian_filter(polar_reconstructed, sigma=3)
    return i, polar_smoothed

print("Reconstructing background gradients...")
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = executor.map(process_single_frame, range(patterns.shape[0]))
    
    for index, bg_frame in results:
        backgrounds[index] = bg_frame
        
print("Done.")

In [ ]:
# calculate and plot master mask for beamstop and zingers
mask_thresh = np.percentile(patterns, 0.1, axis=0)
master_mask = (mask_thresh > 0.1)

In [ ]:
# perform NMF on the backgrounds to extract components for randomized background synthesis

n_frames, height, width = patterns.shape

backgrounds_clean = np.clip(backgrounds, a_min=0, a_max=None)
backgrounds_clean[:, master_mask] = 0

V = backgrounds_clean.reshape(n_frames, height * width)
print(f"V shape: {V.shape}")

n_components = 5 

nmf_model = NMF(
    n_components=n_components, 
    init='nndsvda', 
    max_iter=500, 
    random_state=42
)

print("Fitting NMF model...")
W = nmf_model.fit_transform(V)
H = nmf_model.components_

print(f"Weights matrix W shape: {W.shape}")
print(f"Components matrix H shape: {H.shape}")

In [ ]:
# plot the learned background components
learned_backgrounds = H.reshape(n_components, height, width)

fig, axes = plt.subplots(1, n_components, figsize=(16, 4))
if n_components == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im = ax.imshow(learned_backgrounds[i], cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))
    ax.set_title(f"Component {i+1}")
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
def generate_synthetic_background(components, weights):
    """
    Generates a novel 2D background by jittering real experimental weight states.
    """
    n_frames, n_comps = weights.shape
    
    random_idx = np.random.randint(0, n_frames)
    base_weights = weights[random_idx].copy()
    
    global_exposure_jitter = np.random.uniform(0.9, 1.1)
    independent_jitter = np.random.uniform(0.95, 1.05, size=n_comps)
    
    final_weights = base_weights * global_exposure_jitter * independent_jitter
    
    synthetic_bg = np.zeros_like(components[0])
    for weight, comp in zip(final_weights, components):
        synthetic_bg += weight * comp
        
    return synthetic_bg, final_weights

In [ ]:
# plot an example of a novel synthetic background
novel_bg, applied_weights = generate_synthetic_background(learned_backgrounds, W)

plt.figure(figsize=(6, 6))
plt.title(f"Synthetic BG (Weights: {np.round(applied_weights, 2)})")
plt.imshow(novel_bg, cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))
plt.axis('off')
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()

### 3. Generate Debye-Scherrer rings using pyFAI

First we extract the expected envelope of geometric parameters, resolution in degrees, and maximum instensity. Then we use pyFAI to generate the Debye-Scherrer rings based on these parameters. The generated rings will be used to create synthetic diffraction patterns that mimic the experimental data.

In [ ]:
# calculate distribution envelope for labels per parameter and create a dictionary to store the min, max, diff, and avg values for each parameter
param_names = ["dist", "poni1", "poni2", "rot1", "rot2", "rot3"]
param_stats = {}

for i in range(labels.shape[1]):
    param_values = labels[:, i]
    min_val, max_val = np.min(param_values), np.max(param_values)
    range_val = max_val - min_val
    avg_val = np.mean(param_values)
    std_val = np.std(param_values)

    param_stats[param_names[i]] = {
        "min": min_val, 
        "max": max_val, 
        "range": range_val, 
        "avg": avg_val, 
        "std": std_val
    }
    
    print(f"{param_names[i]}: Min = {min_val}, Max = {max_val}, Range = {range_val}, Avg = {avg_val}, Std = {std_val}")

In [ ]:
# calculate the average peak FWHM and max intensity to inform the generation of synthetic Debye-Scherrer rings from 1D azimuhtal integrations
all_widths_deg = []
all_max_intensities = []

for i in range(patterns.shape[0]):

    pattern_max = np.percentile(patterns[i], 99.9)
    pattern = np.clip(patterns[i], a_min=0, a_max=pattern_max)

    ai = AzimuthalIntegrator(
        dist=labels[i][0], 
        poni1=labels[i][1], 
        poni2=labels[i][2],
        rot1=labels[i][3],
        rot2=labels[i][4],
        rot3=labels[i][5],
        pixel1=PIXEL_SIZE,
        pixel2=PIXEL_SIZE,
        wavelength=WAVELENGTH,
        detector=DETECTOR
    )

    q, intensity = ai.integrate1d(
        pattern, 1000, unit="2th_deg", correctSolidAngle=False, mask=master_mask
    )

    from scipy.signal import find_peaks, peak_widths

    peaks, _ = find_peaks(intensity, height=np.max(intensity)*0.5)
    tallest_peak_idx = np.argmax(intensity[peaks])

    max_val = intensity[peaks][tallest_peak_idx]
    all_max_intensities.append(max_val)

    widths = peak_widths(intensity, peaks, rel_height=0.5)[0]
    tallest_width_bins = widths[tallest_peak_idx]

    dq = q[1] - q[0]
    tallest_width_q = tallest_width_bins * dq

    all_widths_deg.append(tallest_width_q)

# calculate the envelope of the FWHM and max intensity values
avg_fwhm, min_fwhm, max_fwhm = np.mean(all_widths_deg), np.min(all_widths_deg), np.max(all_widths_deg)

avg_Imax, min_Imax, max_Imax = np.mean(all_max_intensities), np.min(all_max_intensities), np.max(all_max_intensities)

print(f"Average FWHM of peaks: {avg_fwhm}")
print(f"Minimum FWHM of peaks: {min_fwhm}")
print(f"Maximum FWHM of peaks: {max_fwhm}")
print(f"Average maximum intensity of peaks: {avg_Imax}")
print(f"Minimum maximum intensity of peaks: {min_Imax}")
print(f"Maximum maximum intensity of peaks: {max_Imax}")

In [ ]:
# sample geometries and peak profiles from the distributions above
resolution = np.random.uniform(min_fwhm, max_fwhm)
Imax = avg_Imax * np.random.uniform(0.95, 1.05)

ai = AzimuthalIntegrator(
    dist=labels[0][0], 
    poni1=labels[0][1],
    poni2=labels[0][2],
    rot1=labels[0][3],
    rot2=labels[0][4],
    rot3=labels[0][5],
    pixel1=PIXEL_SIZE,
    pixel2=PIXEL_SIZE,
    wavelength=WAVELENGTH,
    detector=DETECTOR
)

calibrant = CALIBRANT_FACTORY("alpha_Al2O3")
calibrant.wavelength = WAVELENGTH

rings = calibrant.fake_calibration_image(ai, Imax=Imax, resolution=resolution)

# texture = sum(gaussian_filter(np.random.randn(*rings.shape), sigma=2**(4-i)) * (0.5**i) for i in range(3))
# texture_z = (texture - np.mean(texture)) / np.std(texture)
# texture_strength = 0.1
# texture_map = 1.0 + (texture_strength * texture_z)

# rings = rings * texture_map

In [ ]:
# plot the generated Debye-Scherrer rings
plt.imshow(rings, cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))

In [ ]:
# plot the azimuthal integration of the synthetic rings to verify that the peak widths and intensities are within the expected ranges
q, intensity = ai.integrate1d(
    rings, 1000, unit="2th_deg", correctSolidAngle=False, mask=master_mask
)

plt.plot(q, intensity)

### 4. Synthesize Composite Diffraction Patterns
Now that we have a learned background and generated Debye-Scherrer rings, we can synthesize composite diffraction patterns by adding them together. This serves as a basis for a Poisson photon counting statistics simulation.

In [ ]:
# add the rings to a sampled background
background = generate_synthetic_background(learned_backgrounds, W)[0]
exp_map = rings + background

plt.imshow(exp_map, cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))

In [ ]:
# use the expectation value map as a parameter for a poisson distribution to simulate the photon counting statistics of the final synthetic pattern
synthetic_pattern = np.random.poisson(exp_map)
synthetic_pattern = synthetic_pattern.astype(np.uint32)

# add back the mask
mask_intensity = 4294967295
synthetic_pattern[master_mask] = mask_intensity

In [ ]:
# compare the azimuthal integration of the synthetic pattern to a real experimental pattern
q_synth, intensity_synth = ai.integrate1d(
    synthetic_pattern, 1000, unit="2th_deg", correctSolidAngle=False, mask=master_mask
)

q_real, intensity_real = ai.integrate1d(
    patterns[0], 1000, unit="2th_deg", correctSolidAngle=False, mask=master_mask
)

plt.plot(q_synth, intensity_synth, label='Synthetic Pattern')
plt.plot(q_real, intensity_real, label='Real Pattern', alpha=0.7)
plt.xlabel('2θ (deg)')
plt.ylabel('Intensity')
plt.legend()
plt.show()

In [ ]:
# visually compare the synthetic 2d patternto a real experimental pattern
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Synthetic Pattern")
plt.imshow(synthetic_pattern, cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Real Pattern")
plt.imshow(patterns[0], cmap='viridis', origin='lower', norm=SymLogNorm(linthresh=1))
plt.axis('off')
plt.show()

### 5. Full-Scale Synthetic Diffraction Pattern Generation
Now that we have tuned the parameters for generating synthetic diffraction patterns, we can generate a full-scale synthetic dataset. This dataset will consist of a large number of synthetic diffraction patterns that can be used for training the ML model. A synthetic dataset will be an h5 file containing the learned components and weights of the backgrounds and the pure Debye-Scherrer rings. Upon loading, dynamic randomization will sample a random background, add it to the rings, and simulate the photon counting statistics to generate a synthetic diffraction pattern.

In [ ]:
OUTPUT_PATH = "../datasets/PHAROS_alpha_Al2O3_Eiger2Cdte_1M_5K.h5"
N_TRAIN = 5000
N_VAL = 500


def sample_parameters():
    """Samples geometry, FWHM, and Imax uniformly from the strict experimental envelope."""
    geometry = [
        np.random.uniform(param_stats['dist']['min'], param_stats['dist']['max']),  # dist
        np.random.uniform(param_stats['poni1']['min'], param_stats['poni1']['max']),  # poni1
        np.random.uniform(param_stats['poni2']['min'], param_stats['poni2']['max']),  # poni2
        np.random.uniform(param_stats['rot1']['min'], param_stats['rot1']['max']),  # rot1
        np.random.uniform(param_stats['rot2']['min'], param_stats['rot2']['max']),  # rot2
        0.0                                                             # rot3 strictly masked
    ]
    
    fwhm = np.random.uniform(min_fwhm, max_fwhm)
    Imax = avg_Imax * np.random.uniform(0.95, 1.05)
    
    return geometry, fwhm, Imax

def generate_single_sample(params):
    """Generates the clean Debye-Scherrer rings for a given set of sampled parameters."""
    geometry, fwhm, Imax = params
    dist, poni1, poni2, rot1, rot2, rot3 = geometry
    
    ai = AzimuthalIntegrator(
        dist=dist, 
        poni1=poni1, 
        poni2=poni2, 
        rot1=rot1, 
        rot2=rot2, 
        rot3=rot3, 
        detector=DETECTOR,
        wavelength=WAVELENGTH,
    )

    calibrant = CALIBRANT_FACTORY("alpha_Al2O3")
    calibrant.wavelength = WAVELENGTH
    
    clean_rings = calibrant.fake_calibration_image(ai, Imax=Imax, resolution=fwhm)
    
    return clean_rings.astype(np.float32), np.array(geometry, dtype=np.float32)

def populate_h5_group(hf_group, num_samples, desc="Generating"):
    """Populates an HDF5 group with clean images and labels concurrently."""
    img_shape = master_mask.shape
    dset_images = hf_group.create_dataset("images", shape=(num_samples, *img_shape), dtype=np.float32)
    dset_labels = hf_group.create_dataset("labels", shape=(num_samples, 6), dtype=np.float32)
    
    parameters = [sample_parameters() for _ in range(num_samples)]
    
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(generate_single_sample, parameters), total=num_samples, desc=desc))
        
    for i, (img, lbl) in enumerate(results):
        dset_images[i] = img
        dset_labels[i] = lbl

print(f"\nWriting dataset to {OUTPUT_PATH}")
with h5py.File(OUTPUT_PATH, 'w') as hf:
    
    bg_group = hf.create_group("background_model")
    bg_group.create_dataset("W", data=W.astype(np.float32))                                     # Shape: (n_images, n_components)
    bg_group.create_dataset("H", data=H.astype(np.float32))                                     # Shape: (n_components, valid_pixels)
    bg_group.create_dataset("master_mask", data=master_mask)                                    # Shape: (H, W) boolean
    bg_group.create_dataset("mask_intensity", data=np.array([mask_intensity], dtype=np.uint32)) # Shape: (1,) uint32 

    train_group = hf.create_group("train")
    test_group = hf.create_group("test")
    
    populate_h5_group(train_group, N_TRAIN, desc="Training Set")
    populate_h5_group(test_group, N_VAL, desc="Validation Set")

    centers = np.array([param_stats[name]["avg"] for name in param_names], dtype=np.float32)
    scale_factors = np.array([param_stats[name]["std"] for name in param_names], dtype=np.float32)
    scale_factors[scale_factors == 0.0] = 0.0001  

    norm_group = hf.create_group("normalization")
    norm_group.create_dataset("centers", data=centers)
    norm_group.create_dataset("scale_factors", data=scale_factors)
    
print("Done.")